# 02 — Model Experimentation
### CodeAlpha Sentiment Analysis Project

This notebook is the experimentation scratchpad: preprocessing → feature
engineering → trying multiple models → comparing results → the
cross-domain generalization analysis. The final, cleaned-up versions of
everything here live in `src/` and are run end-to-end via `main.py`;
this notebook is for exploration and sanity-checking along the way.


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data_loader import load_all_domains
from src.preprocessing import preprocess_dataframe
from src.feature_engineering import build_features
from src.train import train_candidates, select_and_save_best
from src.evaluate import evaluate_model
from src.cross_domain_eval import run_cross_domain_matrix
from src.utils import load_pickle

sns.set_theme(style="whitegrid")


## 1. Load data

In [ ]:
raw_df = load_all_domains()
raw_df.head()


## 2. Preprocess text

> First run, once per environment: `python -m src.preprocessing --download-nltk`

In [ ]:
processed_df = preprocess_dataframe(raw_df)
processed_df.to_csv(config.PROCESSED_FILE, index=False)
processed_df[["text", "clean_text", "label", "domain"]].sample(5, random_state=1)


## 3. Feature engineering (TF-IDF) + train/test split

In [ ]:
X_train, X_test, y_train, y_test, train_df, test_df, vectorizer, encoder = build_features(processed_df)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Classes:", list(encoder.classes_))


## 4. Train candidate models

Two linear models on identical features, compared by macro-F1 (see
`docs/methodology.md` for why macro-F1 over accuracy).

In [ ]:
results = train_candidates(X_train, y_train, X_test, y_test)
for name, r in results.items():
    print(f"{name:<22} macro-F1 = {r['macro_f1']:.4f}")


In [ ]:
best_name, best_model, best_score = select_and_save_best(results)
print(f"Selected: {best_name} (macro-F1={best_score:.4f})")


## 5. Evaluate the final model

In [ ]:
model = load_pickle(config.MODEL_PATH)
metrics = evaluate_model(model, X_test, y_test, encoder, test_df)
metrics["overall_accuracy"], metrics["overall_macro_f1"]


In [ ]:
import numpy as np

cm = np.array(metrics["confusion_matrix"])
labels = metrics["label_order"]

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix")
plt.tight_layout()
plt.show()


## 6. Cross-domain generalization analysis

The key experiment: train on one domain, test on every domain (including
itself), and compare against a model trained on all domains pooled
together. See `docs/methodology.md` section 7 for the full reasoning.

In [ ]:
cross_domain_results = run_cross_domain_matrix(processed_df)

matrix_df = pd.DataFrame(cross_domain_results["matrix"])
matrix_df


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(matrix_df, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax,
            cbar_kws={"label": "Macro-F1"})
ax.set_xlabel("Tested on")
ax.set_ylabel("Trained on")
ax.set_title("Cross-domain generalization (macro-F1)")
plt.tight_layout()
plt.show()

print("\nCombined (all-domains) model, evaluated per domain:")
print(cross_domain_results["combined_model_scores"])


## 7. Notes / next steps

Record actual observations here after running against the real
datasets, e.g.:
- Which domain pair had the largest generalization gap?
- Did the combined model recover most of the gap, or did pooling domains
  dilute in-domain performance anywhere?
- Candidate follow-ups: try a transformer-based model (e.g. DistilBERT)
  for comparison; try oversampling the neutral class; try character
  n-grams for the Twitter domain to handle misspellings/slang.